In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image

model_path = 'best_model_yolo.pt'
model = YOLO(model_path)

print(f"Modello caricato da: {model_path}")

Modello caricato da: best_model_yolo.pt


In [3]:
# Funzione per l'inferenza in tempo reale con la webcam
def predict_webcam(camera_id=0, conf_threshold=0.25):
    """
    Esegue l'inferenza in tempo reale utilizzando la webcam
    
    Args:
        camera_id: ID della webcam (di solito 0 per la webcam integrata)
        conf_threshold: Soglia di confidenza per le predizioni
    """
    # Apri la webcam
    cap = cv2.VideoCapture(camera_id)
    
    # Verifica che la webcam sia stata aperta correttamente
    if not cap.isOpened():
        print(f"Errore nell'apertura della webcam con ID: {camera_id}")
        return
    
    print("Inferenza in tempo reale avviata. Premi 'q' per uscire.")
    
    while True:
        # Leggi un frame dalla webcam
        ret, frame = cap.read()
        if not ret:
            break
        
        # Esegui la predizione
        results = model(frame, conf=conf_threshold)
        result_frame = results[0].plot()
        
        # Mostra il frame con le predizioni
        cv2.imshow('Rilevamento Pistole in Tempo Reale', result_frame)
        
        # Esci se viene premuto 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    # Rilascia le risorse
    cap.release()
    cv2.destroyAllWindows()
    
    print("Inferenza in tempo reale terminata.")

In [4]:
def predict_folder(folder_path, conf_threshold=0.25, save_dir='predictions'):
    """
    Esegue l'inferenza su tutte le immagini in una cartella
    
    Args:
        folder_path: Percorso della cartella contenente le immagini
        conf_threshold: Soglia di confidenza per le predizioni
        save_dir: Cartella dove salvare i risultati
    """
    # Crea la cartella per salvare i risultati se non esiste
    os.makedirs(save_dir, exist_ok=True)
    
    # Estensioni valide per le immagini
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    
    # Lista tutte le immagini nella cartella
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]
    
    if not image_files:
        print(f"Nessuna immagine trovata in: {folder_path}")
        return
    
    print(f"Trovate {len(image_files)} immagini. Inizio inferenza...")
    
    for img_file in image_files:
        # Costruisci il percorso completo
        img_path = os.path.join(folder_path, img_file)
        
        # Carica l'immagine
        img = cv2.imread(img_path)
        
        # Esegui la predizione
        results = model(img, conf=conf_threshold)
        result_img = results[0].plot()
        
        # Salva l'immagine con le predizioni
        save_path = os.path.join(save_dir, f'pred_{img_file}')
        cv2.imwrite(save_path, result_img)
        
        # Stampa le informazioni sulle predizioni
        boxes = results[0].boxes
        if len(boxes) > 0:
            print(f"\nRilevamenti in {img_file}:")
            for box in boxes:
                conf = box.conf.item()
                print(f"- Confidenza: {conf:.2f}")
    
    print(f"\nInferenza completata. Risultati salvati in: {save_dir}")

In [6]:
# Avvia l'inferenza in tempo reale con la webcam
predict_folder(folder_path='Dataset/yolo/test')
# predict_webcam(camera_id=0, conf_threshold=0.3)

Trovate 20 immagini. Inizio inferenza...

0: 448x640 2 pistols, 196.0ms
Speed: 9.8ms preprocess, 196.0ms inference, 7.5ms postprocess per image at shape (1, 3, 448, 640)

Rilevamenti in test (1).jpeg:
- Confidenza: 0.81
- Confidenza: 0.30

0: 480x640 1 pistol, 117.4ms
Speed: 3.5ms preprocess, 117.4ms inference, 31.0ms postprocess per image at shape (1, 3, 480, 640)

Rilevamenti in test (1).jpg:
- Confidenza: 0.35

0: 480x640 1 pistol, 106.0ms
Speed: 3.2ms preprocess, 106.0ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

Rilevamenti in test (10).jpg:
- Confidenza: 0.87

0: 448x640 2 pistols, 120.2ms
Speed: 3.9ms preprocess, 120.2ms inference, 0.7ms postprocess per image at shape (1, 3, 448, 640)

Rilevamenti in test (11).jpg:
- Confidenza: 0.32
- Confidenza: 0.29

0: 384x640 4 pistols, 176.7ms
Speed: 2.2ms preprocess, 176.7ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

Rilevamenti in test (12).jpg:
- Confidenza: 0.93
- Confidenza: 0.91
- Confide